In [ ]:
import networkx as nx
import numpy as np
import plotly.graph_objects as go

from ramsey.RColoring import RColoring
from ramsey.RArchive import RSQLiteArchive


def plot_coloring_3d(
    coloring: RColoring,
    *,
    layout_seed: int = 43,
    layout_iterations: int = 300,
    edge_opacity: float = 0.12,
    title: str | None = None,
) -> go.Figure:
    """
    Create an interactive 3D visualization of a two-coloring.

    Blue and red edges are separate legend layers. Vertices are
    colored by blue degree minus red degree.
    """
    graph = coloring.graph
    edges = graph.edges
    colors = coloring.colors

    n_vertices = graph.problem.n_vertices

    blue_mask = colors == 1
    red_mask = colors == 0

    blue_edges = edges[blue_mask]
    red_edges = edges[red_mask]

    # Use the blue graph to calculate a force-directed layout.
    layout_graph = nx.Graph()

    layout_graph.add_nodes_from(range(n_vertices))

    layout_graph.add_edges_from(
        (
            int(i),
            int(j),
        )
        for i, j in blue_edges
    )

    positions = nx.spring_layout(
        layout_graph,
        dim=3,
        seed=layout_seed,
        iterations=layout_iterations,
        k=1.0 / np.sqrt(n_vertices),
    )

    coordinates = np.asarray(
        [positions[vertex] for vertex in range(n_vertices)],
        dtype=np.float64,
    )

    blue_degrees = np.zeros(
        n_vertices,
        dtype=np.int32,
    )

    np.add.at(
        blue_degrees,
        blue_edges[:, 0],
        1,
    )

    np.add.at(
        blue_degrees,
        blue_edges[:, 1],
        1,
    )

    red_degrees = n_vertices - 1 - blue_degrees

    net_color_counts = blue_degrees - red_degrees

    def make_edge_trace(
        selected_edges: np.ndarray,
        *,
        name: str,
        color: str,
    ) -> go.Scatter3d:
        x_coordinates: list[float | None] = []
        y_coordinates: list[float | None] = []
        z_coordinates: list[float | None] = []

        for i, j in selected_edges:
            i = int(i)
            j = int(j)

            x_coordinates.extend(
                (
                    coordinates[i, 0],
                    coordinates[j, 0],
                    None,
                )
            )

            y_coordinates.extend(
                (
                    coordinates[i, 1],
                    coordinates[j, 1],
                    None,
                )
            )

            z_coordinates.extend(
                (
                    coordinates[i, 2],
                    coordinates[j, 2],
                    None,
                )
            )

        return go.Scatter3d(
            x=x_coordinates,
            y=y_coordinates,
            z=z_coordinates,
            mode="lines",
            name=name,
            opacity=0.25,
            line={
                "color": color,
                "width": 1,
            },
            hoverinfo="skip",
        )

    blue_trace = make_edge_trace(
        blue_edges,
        name="Blue edges",
        color="blue",
    )

    red_trace = make_edge_trace(
        red_edges,
        name="Red edges",
        color="red",
    )

    maximum_imbalance = max(
        1,
        int(np.abs(net_color_counts).max()),
    )

    hover_text = [
        (
            f"Vertex {vertex}<br>"
            f"Blue degree: "
            f"{blue_degrees[vertex]}<br>"
            f"Red degree: "
            f"{red_degrees[vertex]}<br>"
            f"Net color: "
            f"{net_color_counts[vertex]:+d}"
        )
        for vertex in range(n_vertices)
    ]

    vertex_trace = go.Scatter3d(
        x=coordinates[:, 0],
        y=coordinates[:, 1],
        z=coordinates[:, 2],
        mode="markers+text",
        name="Vertices",
        text=[str(vertex) for vertex in range(n_vertices)],
        textposition="top center",
        hovertext=hover_text,
        hoverinfo="text",
        marker={
            "size": 7,
            "color": net_color_counts,
            "colorscale": "RdBu",
            "cmin": -maximum_imbalance,
            "cmax": maximum_imbalance,
            "line": {
                "color": "black",
                "width": 1,
            },
            "colorbar": {
                "title": "Blue − red<br>degree",
            },
        },
    )

    if title is None:
        title = f"Interactive K{n_vertices} " "red-blue coloring"

    figure = go.Figure(
        data=[
            red_trace,
            blue_trace,
            vertex_trace,
        ]
    )

    figure.update_layout(
        title=title,
        autosize=True,
        showlegend=False,
        scene={
            "xaxis": {
                "visible": False,
            },
            "yaxis": {
                "visible": False,
            },
            "zaxis": {
                "visible": False,
            },
            "aspectmode": "data",
            "bgcolor": "rgb(245, 245, 245)",
        },
        margin={
            "l": 0,
            "r": 0,
            "b": 0,
            "t": 40,
        },
    )

    return figure

In [12]:
# Load an Archived K43 Coloring
from pathlib import Path

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

DATABASE_PATH = (
    project_root
    / "data"
    / "ramsey_colorings.sqlite3"
)

problem = RProblem.r55(
    n_vertices=43,
)

graph = RGraph(problem)

archive = RSQLiteArchive(
    DATABASE_PATH
)

archive_best = archive.best_score(
    graph
)

best_record = archive.best_colorings(
    limit=1,
    graph=graph,
)[0]

archived = archive.load_coloring(
    best_record.coloring_id,
    graph,
)

coloring = archived.coloring

state = RSearchState(
    coloring
)

print("Archive ID:", best_record.coloring_id)
print("Score:", state.score)
print("Run:", best_record.run_name)
print("Iteration:", best_record.iteration)


Archive ID: 1735
Score: 278
Run: greedy-tabu-sub-400-phase-3-seeds-002
Iteration: 254


In [ ]:
# 3D Graph plot
figure = plot_coloring_3d(
    coloring,
    title="Random K43 coloring",
)

figure.show(
    renderer="browser",
    config={
        "responsive": True,
    },
)

In [18]:
# 2D Circular K43 Graph Plot
#
# Edge intensity = number of monochromatic K5s containing the edge.
# Vertex intensity = number of monochromatic K5s containing the vertex.

import numpy as np
import plotly.graph_objects as go


N_VERTICES = graph.problem.n_vertices

vertex_labels = np.arange(
    N_VERTICES,
    dtype=np.int32,
)

angles = (
    np.pi / 2
    - 2 * np.pi * vertex_labels / N_VERTICES
)

x = np.cos(angles)
y = np.sin(angles)


# ------------------------------------------------------------
# Edge monochromatic-K5 participation
# ------------------------------------------------------------

profiles = state.action_profiles

edge_violation_loads = (
    profiles[:, 0].astype(np.int32)
    + profiles[:, -1].astype(np.int32)
)

maximum_edge_load = int(
    edge_violation_loads.max()
)


# ------------------------------------------------------------
# Vertex monochromatic-K5 participation
# ------------------------------------------------------------

#
# Sum the violation load of every incident edge.
#
# Each monochromatic K5 containing a vertex contributes four
# such edges, so divide the result by four.
#

vertex_edge_load_sums = np.zeros(
    N_VERTICES,
    dtype=np.int32,
)

np.add.at(
    vertex_edge_load_sums,
    graph.edges[:, 0],
    edge_violation_loads,
)

np.add.at(
    vertex_edge_load_sums,
    graph.edges[:, 1],
    edge_violation_loads,
)

if np.any(vertex_edge_load_sums % 4 != 0):
    raise RuntimeError(
        "Vertex participation counts are not divisible by four."
    )

vertex_violation_loads = (
    vertex_edge_load_sums // 4
)

maximum_vertex_load = int(
    vertex_violation_loads.max()
)


# ------------------------------------------------------------
# Vertex blue/red degrees
# ------------------------------------------------------------

blue_degrees = np.zeros(
    N_VERTICES,
    dtype=np.int32,
)

for edge_index, (u, v) in enumerate(graph.edges):
    if coloring.colors[edge_index] == 1:
        blue_degrees[u] += 1
        blue_degrees[v] += 1

red_degrees = (
    N_VERTICES - 1 - blue_degrees
)

imbalances = (
    blue_degrees - red_degrees
)


# ------------------------------------------------------------
# Verify the participation identities
# ------------------------------------------------------------

expected_edge_total = (
    10 * state.score
)

expected_vertex_total = (
    5 * state.score
)

actual_edge_total = int(
    edge_violation_loads.sum()
)

actual_vertex_total = int(
    vertex_violation_loads.sum()
)

print("Score:", state.score)

print(
    "Edge participation:",
    actual_edge_total,
    "==",
    expected_edge_total,
)

print(
    "Vertex participation:",
    actual_vertex_total,
    "==",
    expected_vertex_total,
)

print(
    "Maximum edge load:",
    maximum_edge_load,
)

print(
    "Maximum vertex load:",
    maximum_vertex_load,
)


# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------

figure = go.Figure()


# ------------------------------------------------------------
# Edges
#
# Plot edges in buckets according to their K5 violation load.
#
# Plotly applies one opacity/width to an entire line trace,
# so bucketing lets heavily implicated edges become both darker
# and thicker without creating 903 individual traces.
# ------------------------------------------------------------

unique_edge_loads = np.unique(
    edge_violation_loads
)

for load in unique_edge_loads:
    load = int(load)

    if maximum_edge_load > 0:
        strength = (
            load / maximum_edge_load
        ) ** 0.70
    else:
        strength = 0.0

    opacity = (
        0.025
        + 0.90 * strength
    )

    width = (
        0.35
        + 3.0 * strength
    )

    for edge_color, plot_color in (
        (0, "red"),
        (1, "blue"),
    ):
        mask = (
            (coloring.colors == edge_color)
            & (edge_violation_loads == load)
        )

        selected_edges = graph.edges[
            mask
        ]

        if len(selected_edges) == 0:
            continue

        edge_x = []
        edge_y = []

        for u, v in selected_edges:
            u = int(u)
            v = int(v)

            edge_x.extend(
                [x[u], x[v], None]
            )

            edge_y.extend(
                [y[u], y[v], None]
            )

        figure.add_trace(
            go.Scatter(
                x=edge_x,
                y=edge_y,
                mode="lines",
                line=dict(
                    color=plot_color,
                    width=width,
                ),
                opacity=opacity,
                hoverinfo="skip",
                showlegend=False,
            )
        )


# ------------------------------------------------------------
# Vertices
#
# Vertex color now represents monochromatic-K5 participation.
# ------------------------------------------------------------

hover_text = [
    (
        f"Vertex {vertex}<br>"
        f"Monochromatic K5s: {load}<br>"
        f"Blue degree: {blue}<br>"
        f"Red degree: {red}<br>"
        f"Imbalance: {imbalance:+d}"
    )
    for (
        vertex,
        load,
        blue,
        red,
        imbalance,
    )
    in zip(
        vertex_labels,
        vertex_violation_loads,
        blue_degrees,
        red_degrees,
        imbalances,
    )
]

figure.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode="markers+text",
        text=[
            str(vertex)
            for vertex in vertex_labels
        ],
        textposition="top center",
        hovertext=hover_text,
        hoverinfo="text",
        marker=dict(
            size=16,
            color=vertex_violation_loads,
            colorscale="YlOrRd",
            cmin=0,
            cmax=max(
                1,
                maximum_vertex_load,
            ),
            line=dict(
                color="#202020",
                width=1,
            ),
            colorbar=dict(
                title="Mono K5s<br>per vertex",
            ),
        ),
        showlegend=False,
    )
)


# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

figure.update_layout(
    title=(
        f"Archived K{N_VERTICES} — Score {state.score}"
        "<br>"
        "<sup>Edge intensity = monochromatic K5 participation; "
        "vertex intensity = monochromatic K5 participation</sup>"
    ),
    autosize=True,
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(
        l=20,
        r=20,
        t=90,
        b=20,
    ),
    xaxis=dict(
        visible=False,
        scaleanchor="y",
        scaleratio=1,
    ),
    yaxis=dict(
        visible=False,
    ),
)

figure.show(
    renderer="browser",
    config={
        "responsive": True,
        "scrollZoom": True,
    },
)

Score: 278
Edge participation: 2780 == 2780
Vertex participation: 1390 == 1390
Maximum edge load: 12
Maximum vertex load: 44
